In [7]:
import pandas as pd
import numpy as np

path = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\AccuracyMeans.csv"
df = pd.read_csv(path)

id_vars = ['participant','gender','negative_affect','positive_affect']
value_vars = [c for c in df.columns if c not in id_vars]
accuracyLong = pd.melt(df, id_vars=id_vars, value_vars=value_vars, var_name='emotion', value_name='value')
accuracyLong['value'] = pd.to_numeric(accuracyLong['value'], errors='coerce')
accuracyLong['emotion'] = accuracyLong['emotion'].astype(str).str.replace(r'_acc$','',regex=True).str.capitalize()
stats = accuracyLong.groupby('emotion', observed=True)['value'].agg(['mean','std','count']).reset_index()

# Turn this into a dataframe
stats_df = stats[['emotion','mean','std']]

# Rename columns
stats_df = stats_df.rename(columns={'emotion': 'Expression', 'mean': 'Mean', 'std': 'S.D.'})

In [8]:
# Path to the RADIATE CSV file in this workspace
path_radiate = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\RADIATE_mean_prop_correct.csv"
radiate = pd.read_csv(path_radiate)

# Remove all the open expressions except for angry and fear and surprise
radiate = radiate[~radiate['Expression'].str.contains('open') | radiate['Expression'].str.contains('Angry|Fear|Surprise')]

# remove rows 0 and 6
radiate = radiate.drop([0, 2, 6, 9])

# now remove the open\closed strings from the Expression column
radiate['Expression'] = radiate['Expression'].str.replace(' (open)', '').str.replace(' (closed)', '').str.strip()

In [9]:
# Combine the two dataframes (stats_df from AccuracyMeans and radiate from RADIATE file)
# Use descriptive suffixes instead of the default _x/_y so it's clear which dataset each column came from
combined = pd.merge(stats_df, radiate, on='Expression', how='inner', suffixes=('_UBIVFED','_RADIATE'))

# Outer merge with indicator to list any expressions that exist in only one dataframe (also use same suffixes)
merged_with_indicator = pd.merge(stats_df, radiate, on='Expression', how='outer', indicator=True, suffixes=('_UBIVFED','_RADIATE'))
left_only = merged_with_indicator[merged_with_indicator['_merge'] == 'left_only'][['Expression']]
right_only = merged_with_indicator[merged_with_indicator['_merge'] == 'right_only'][['Expression']]

# Replace the Happy expression to Joy (post-merge to affect both datasets)
combined.loc[combined['Expression'] == 'Happy', 'Expression'] = 'Joy'

# Identify expected mean/sd columns after merge
mean_col_ub = 'Mean_UBIVFED'
sd_col_ub = 'S.D._UBIVFED'
mean_col_rad = 'Mean_RADIATE'
sd_col_rad = 'S.D._RADIATE'

# Round numerical columns (means & sds) for consistent formatting
for c in [mean_col_ub, sd_col_ub, mean_col_rad, sd_col_rad]:
    if c in combined.columns:
        combined[c] = combined[c].astype(float).round(2)

# Build display columns with "mean (sd)" formatting
combined['UBIVFED'] = combined.apply(lambda r: f"{r[mean_col_ub]:.2f} ({r[sd_col_ub]:.2f})", axis=1)
combined['RADIATE'] = combined.apply(lambda r: f"{r[mean_col_rad]:.2f} ({r[sd_col_rad]:.2f})", axis=1)

# Row-wise mean accuracy (average of means only)
combined['Mean accuracy'] = ((combined[mean_col_ub] + combined[mean_col_rad]) / 2).round(2)
combined['Mean accuracy'] = combined['Mean accuracy'].map(lambda v: f"{v:.2f}")

# Create final table with only required columns
final_table = combined[['Expression','UBIVFED','RADIATE','Mean accuracy']].copy()
# Rename UBIVFED/RADIATE columns to requested headers
final_table = final_table.rename(columns={'UBIVFED':'Mean (sd) UBIVFED','RADIATE':'Mean (sd) RADIATE'})

# Bottom summary row: overall mean (sd) per dataset + overall mean accuracy
overall_mean_ub = combined[mean_col_ub].mean()
overall_sd_ub = 0.076
overall_mean_rad = combined[mean_col_rad].mean()
overall_sd_rad = 0.19
overall_mean_accuracy = ((overall_mean_ub + overall_mean_rad) / 2)

summary_row = {
    'Expression': 'Overall',
    'Mean (sd) UBIVFED': f"{overall_mean_ub:.2f} ({overall_sd_ub:.2f})",
    'Mean (sd) RADIATE': f"{overall_mean_rad:.2f} ({overall_sd_rad:.2f})",
    'Mean accuracy': f"{overall_mean_accuracy:.2f}"
}
final_table = pd.concat([final_table, pd.DataFrame([summary_row])], ignore_index=True)

# Convert to LaTeX (no index)
latex_table = final_table.to_latex(index=False,
                                   caption='Comparison of Emotion Recognition Accuracy: UBIVFED vs RADIATE',
                                   label='tab:accuracy_comparison',
                                   position='h')

# Remove underscores for cleaner appearance
latex_table = latex_table.replace('_', ' ')

# Save LaTeX table to file
output_path = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\accuracy_comparison_table.tex"
with open(output_path, 'w') as f:
    f.write(latex_table)

In [10]:
# read the comparison CSV (path from the original R script)
path_comp = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\Master Compiled Doc - Cleaned - Comparisons.csv"
comparison = pd.read_csv(path_comp)

# split by condition
comparison_CC = comparison[comparison['Condition'] == 'CC']
comparison_IC = comparison[comparison['Condition'] == 'IC']

# emotions of interest (same order as the R code)
emotions = ['Angry','Disgust','Fear','Happy','Neutral','Sad','Surprise']

def summarize_group(df, emotions):
    rows = []
    for e in emotions:
        sub = df[df['VirtEmotionID'] == e]
        rows.append({
            'Emotion': e,
            'scaled_mean': sub['scaled'].mean(),
            'scaled_sd': sub['scaled'].std(),
            'slider_mean': sub['slider.response'].mean(),
            'slider_sd': sub['slider.response'].std(),
            'n': len(sub)
        })
    return pd.DataFrame(rows)

cc_summary = summarize_group(comparison_CC, emotions)
ic_summary = summarize_group(comparison_IC, emotions)

# show results
print("Congruent (CC):")
print(cc_summary.to_string(index=False))
print("\nIncongruent (IC):")
print(ic_summary.to_string(index=False))

Congruent (CC):
 Emotion  scaled_mean  scaled_sd  slider_mean  slider_sd   n
   Angry     1.524194   1.586818     5.524194   1.586818 620
 Disgust     1.796774   1.522260     5.796774   1.522260 620
    Fear     1.812903   1.587663     5.812903   1.587663 620
   Happy     2.261290   1.120863     6.261290   1.120863 620
 Neutral     2.262903   1.153870     6.262903   1.153870 620
     Sad     1.912903   1.463225     5.912903   1.463225 620
Surprise     2.335484   1.087615     6.335484   1.087615 620

Incongruent (IC):
 Emotion  scaled_mean  scaled_sd  slider_mean  slider_sd   n
   Angry     1.169355   1.765409     5.169355   1.765409 620
 Disgust     1.416129   1.680756     5.416129   1.680756 620
    Fear     1.233871   1.814984     5.233871   1.814984 620
   Happy     1.969355   1.317472     5.969355   1.317472 620
 Neutral     1.985484   1.468500     5.985484   1.468500 620
     Sad     1.917742   1.430571     5.917742   1.430571 620
Surprise     2.011290   1.391714     6.011290   1.

In [11]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# from matplotlib.patches import Patch
# from matplotlib.lines import Line2D

# # Path to the RatingCountsMaster CSV in this workspace
# path_counts = r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\processed_data\zedd_data\RatingCountsMaster.csv"
# df_counts = pd.read_csv(path_counts)

# # ---- Recreate the similarity means plot as a single grouped plot (two bars per emotion) ----
# # R values copied directly from the provided script
# targetEmotion = ['Anger','Disgust','Fear','Joy','Neutral','Sad','Surprise','Anger','Disgust','Fear','Joy','Neutral','Sad','Surprise']
# condition = ['CC']*7 + ['IC']*7
# sliderMean = [5.524194, 5.796774, 5.812903, 6.26129, 6.262903, 5.912903, 6.335484, 5.169355, 5.416129, 5.233871, 5.969355, 5.985484, 5.917742, 6.01129]
# scaledMean = [1.524194, 1.796774, 1.812903, 2.26129, 2.262903, 1.912903, 2.335484, 1.169355, 1.416129, 1.233871, 1.969355, 1.985484, 1.917742, 2.01129]
# sD = [1.586818, 1.52226, 1.587663, 1.120863, 1.15387, 1.463225, 1.087615, 1.765409, 1.680756, 1.814984, 1.317472, 1.4685, 1.430571, 1.391714]

# similarityMeans = pd.DataFrame({'targetEmotion': targetEmotion, 'condition': condition, 'sliderMean': sliderMean, 'scaledMean': scaledMean, 'sD': sD})

# # Colors from the R script (strip alpha if present) -- mapped to the 7 emotions in order
# hex_colors = ['#DC0000FF', '#4DBBD5FF', '#00A087FF', '#3C5488FF', '#F39B7FFF', '#8491B4FF', '#91D1C2FF']
# # convert to #RRGGBB by taking the first 7 characters (safe for these strings)
# colors = [h[:7] for h in hex_colors]

# # Unique emotions in the desired order
# unique_emotions = ['Anger','Disgust','Fear','Joy','Neutral','Sad','Surprise']
# n = len(unique_emotions)

# # prepare CC/IC arrays in emotion order
# cc = similarityMeans[similarityMeans['condition']=='CC'].set_index('targetEmotion').reindex(unique_emotions)
# ic = similarityMeans[similarityMeans['condition']=='IC'].set_index('targetEmotion').reindex(unique_emotions)
# cc_means = cc['scaledMean'].values
# ic_means = ic['scaledMean'].values
# cc_sds = cc['sD'].values
# ic_sds = ic['sD'].values

# # --- Publication-style plotting parameters ---
# sns.set_style('whitegrid')
# plt.rcParams.update({
#     'font.size': 16,
#     'axes.titlesize': 22,
#     'axes.labelsize': 18,
#     'xtick.labelsize': 16,
#     'ytick.labelsize': 16,
#     'legend.fontsize': 14,
#     'figure.dpi': 150,
# })

# # plotting grouped bars (two bars per emotion)
# x = np.arange(n)
# width = 0.32
# fig, ax = plt.subplots(figsize=(11,4.2))
# bar_containers = []
# for i in range(n):
#     # draw CC and IC bars without error bars
#     b1 = ax.bar(x[i]-width/2, cc_means[i], width, color=colors[i], edgecolor='black', linewidth=0.4)
#     b2 = ax.bar(x[i]+width/2, ic_means[i], width, color=colors[i], edgecolor='black', linewidth=0.4, alpha=0.95)
#     bar_containers.append((b1,b2))

# # labels and title
# ax.set_xticks(x)
# ax.set_xticklabels(unique_emotions, fontsize=16)
# ax.set_ylabel('Degree of Similarity', fontsize=18)
# ax.set_title('Similarity ratings across conditions', fontsize=22)
# ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

# # small tick labels under each pair to indicate condition (subtle)
# for i in range(n):
#     # place CC/IC labels below the x-axis using axis transform (axes coords for y)
#     # moved further down to avoid overlap and improve readability
#     ax.text(x[i]-width/2, -0.14, 'CC', ha='center', va='top', fontsize=13, color='black', transform=ax.get_xaxis_transform())
#     ax.text(x[i]+width/2, -0.14, 'IC', ha='center', va='top', fontsize=13, color='black', transform=ax.get_xaxis_transform())

# # legend (emotion colors)
# legend_patches = [Patch(facecolor=colors[i], edgecolor='black', label=unique_emotions[i]) for i in range(n)]
# ns_line = Line2D([0], [0], color='black', lw=2, label='n.s.')
# legend_handles = [ns_line] + legend_patches
# ax.legend(handles=legend_handles, bbox_to_anchor=(1.02, 0.92), loc='upper left', frameon=False)

# # move 'n.s.' annotation over 'Sad' instead of 'Anger'
# sad_idx = unique_emotions.index('Sad')
# # recalculate ymax using only means (no error bars)
# ymax = max(np.nanmax(cc_means), np.nanmax(ic_means))
# line_offset = 0.35  # vertical offset above tallest bar
# line_y = ymax + line_offset
# ax.plot([x[sad_idx]-width/2, x[sad_idx]+width/2], [line_y, line_y], color='black', lw=1.6)
# ax.text(x[sad_idx], line_y + 0.06, 'n.s.', ha='center', va='bottom', fontsize=14)

# # tidy up limits and layout
# ax.set_ylim(bottom=0, top=line_y + 0.5)
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)
# # make room at the bottom so the CC/IC labels are not clipped
# # increased bottom margin to accommodate the lowered CC/IC labels
# fig.subplots_adjust(bottom=0.16)
# fig.tight_layout()
# plt.savefig(r"C:\Users\super\OneDrive - Ontario Tech University\fNIRS_Emotions\plots\zedd_figures\similarity_means_grouped_plot.png", dpi=300)
# plt.close()